In [1]:
from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd

from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap

from bst_bev.maps.bst_map_index import BSTMapIndex
from bst_bev.query.fixed_zoom import FixedZoomRegionCover


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATAROOT = PROJECT_ROOT / "data" / "nuscenes"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RAW_OUTPUT = (
    OUTPUT_DIR
    / "direct_query_all_samples.csv"
)

SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "direct_query_summary.csv"
)

ZOOMS = [23, 24]
RADIUS_M = 30.0

MAP_LAYERS = (
    "lane_divider",
    "road_divider",
    "ped_crossing",
)

In [2]:
nusc = NuScenes(
    version="v1.0-mini",
    dataroot=str(DATAROOT),
    verbose=False,
)

print("Scenes:", len(nusc.scene))
print("Samples:", len(nusc.sample))

Scenes: 10
Samples: 404


In [3]:
def get_sample_context(
    nusc,
    sample,
):
    scene = nusc.get(
        "scene",
        sample["scene_token"],
    )

    log = nusc.get(
        "log",
        scene["log_token"],
    )

    location = log["location"]

    lidar = nusc.get(
        "sample_data",
        sample["data"]["LIDAR_TOP"],
    )

    ego_pose = nusc.get(
        "ego_pose",
        lidar["ego_pose_token"],
    )

    ego_x, ego_y, _ = (
        ego_pose["translation"]
    )

    return {
        "scene": scene["name"],
        "location": location,
        "ego_x": ego_x,
        "ego_y": ego_y,
    }

In [4]:
sample_contexts = []

for sample in nusc.sample:

    context = get_sample_context(
        nusc,
        sample,
    )

    sample_contexts.append({
        "sample_token": sample["token"],
        "timestamp": sample["timestamp"],
        **context,
    })

samples_df = pd.DataFrame(
    sample_contexts
)

samples_df.head()

,sample_token,timestamp,scene,location,ego_x,ego_y
0,ca9a282c9e77460f8360f564131a8af5,1532402927647951,scene-0061,singapore-onenorth,411.303935,1180.890379
1,39586f9d59004284a7114a68825e8eec,1532402928147847,scene-0061,singapore-onenorth,409.743152,1176.676973
2,356d81f38dd9473ba590f39e266f54e5,1532402928698048,scene-0061,singapore-onenorth,408.101879,1172.151578
3,e0845f5322254dafadbbed75aaa07969,1532402929197353,scene-0061,singapore-onenorth,406.726006,1168.211287
4,c923fe08b2ff4e27975d2bf30934383b,1532402929697797,scene-0061,singapore-onenorth,405.409406,1164.354227


In [5]:
locations = sorted(
    samples_df["location"].unique()
)

nusc_maps = {
    location: NuScenesMap(
        dataroot=str(DATAROOT),
        map_name=location,
    )
    for location in locations
}

locations

['boston-seaport',
 'singapore-hollandvillage',
 'singapore-onenorth',
 'singapore-queenstown']

In [6]:
def get_ground_truth_tokens(
    nusc_map,
    ego_x,
    ego_y,
    radius=RADIUS_M,
):
    patch = (
        ego_x - radius,
        ego_y - radius,
        ego_x + radius,
        ego_y + radius,
    )

    records = (
        nusc_map.get_records_in_patch(
            patch,
            list(MAP_LAYERS),
            mode="intersect",
        )
    )

    return {
        (layer, token)
        for layer, tokens
        in records.items()
        for token in tokens
    }

ground_truth_cache = {}

for _, row in samples_df.iterrows():

    gt = get_ground_truth_tokens(
        nusc_map=nusc_maps[
            row["location"]
        ],
        ego_x=row["ego_x"],
        ego_y=row["ego_y"],
    )

    ground_truth_cache[
        row["sample_token"]
    ] = gt

print(
    "GT samples:",
    len(ground_truth_cache),
)

GT samples: 404


In [7]:
def evaluate_queries(
    index,
    region_cover,
    row,
    gt_tokens,
):
    # ==================================================
    # Sampling query
    # ==================================================

    start = time.perf_counter()

    sampling = index.query_patch(
        center_x=row["ego_x"],
        center_y=row["ego_y"],
        radius=RADIUS_M,
        step_m=None,
    )

    sampling_time_ms = (
        time.perf_counter() - start
    ) * 1000.0

    sampling_tokens = sampling.tokens

    # ==================================================
    # Direct cell cover
    # ==================================================

    start = time.perf_counter()

    direct_cover = region_cover.cover(
        center_x=row["ego_x"],
        center_y=row["ego_y"],
        radius=RADIUS_M,
    )

    direct_cover_time_ms = (
        time.perf_counter() - start
    ) * 1000.0

    # ==================================================
    # Direct index lookup
    # ==================================================

    start = time.perf_counter()

    direct_tokens = (
        index.query_cell_tokens(
            direct_cover.keys
        )
    )

    direct_lookup_time_ms = (
        time.perf_counter() - start
    ) * 1000.0

    direct_total_time_ms = (
        direct_cover_time_ms
        + direct_lookup_time_ms
    )

    # ==================================================
    # Method agreement
    # ==================================================

    intersection = (
        sampling_tokens
        & direct_tokens
    )

    union = (
        sampling_tokens
        | direct_tokens
    )

    jaccard = (
        len(intersection)
        / len(union)
        if union
        else 1.0
    )

    same_candidates = (
        sampling_tokens
        == direct_tokens
    )

    # ==================================================
    # Direct method vs GT
    # ==================================================

    tp = len(
        direct_tokens & gt_tokens
    )

    fp = len(
        direct_tokens - gt_tokens
    )

    fn = len(
        gt_tokens - direct_tokens
    )

    precision = (
        tp / (tp + fp)
        if tp + fp
        else np.nan
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else np.nan
    )

    return {
        "sampling_cells":
            sampling.cell_count,

        "direct_cells":
            direct_cover.cell_count,

        "same_cell_count":
            (
                sampling.cell_count
                == direct_cover.cell_count
            ),

        "sampling_candidates":
            len(sampling_tokens),

        "direct_candidates":
            len(direct_tokens),

        "same_candidates":
            same_candidates,

        "candidate_jaccard":
            jaccard,

        "sampling_only_count":
            len(
                sampling_tokens
                - direct_tokens
            ),

        "direct_only_count":
            len(
                direct_tokens
                - sampling_tokens
            ),

        "tp": tp,
        "fp": fp,
        "fn": fn,

        "precision": precision,
        "recall": recall,

        "sampling_time_ms":
            sampling_time_ms,

        "direct_cover_time_ms":
            direct_cover_time_ms,

        "direct_lookup_time_ms":
            direct_lookup_time_ms,

        "direct_total_time_ms":
            direct_total_time_ms,

        "speedup": (
            sampling_time_ms
            / direct_total_time_ms
            if direct_total_time_ms > 0
            else np.nan
        ),
    }

In [8]:
results = []

experiment_start = (
    time.perf_counter()
)


for zoom in ZOOMS:

    print()
    print("=" * 60)
    print(f"ZOOM {zoom}")
    print("=" * 60)

    for location in locations:

        print(
            f"{location}: building index..."
        )

        index = BSTMapIndex(
            nusc_map=nusc_maps[location],
            location=location,
            zoom=zoom,
            index_spacing_m=1.0,
            adaptive_index_spacing=False,
        )

        index.build()

        region_cover = (
            FixedZoomRegionCover(
                location=location,
                zoom=zoom,
            )
        )

        location_samples = (
            samples_df[
                samples_df["location"]
                == location
            ]
        )

        for _, row in (
            location_samples.iterrows()
        ):

            gt_tokens = (
                ground_truth_cache[
                    row["sample_token"]
                ]
            )

            metrics = evaluate_queries(
                index=index,
                region_cover=region_cover,
                row=row,
                gt_tokens=gt_tokens,
            )

            results.append({
                "zoom": zoom,

                "scene": row["scene"],

                "sample_token":
                    row["sample_token"],

                "location": location,

                "ego_x": row["ego_x"],
                "ego_y": row["ego_y"],

                "ground_truth":
                    len(gt_tokens),

                **metrics,
            })

        del index
        gc.collect()

    # Checkpoint after each zoom.
    results_df = pd.DataFrame(
        results
    )

    results_df.to_csv(
        RAW_OUTPUT,
        index=False,
    )

    print(
        f"Checkpoint written: "
        f"{RAW_OUTPUT}"
    )


elapsed = (
    time.perf_counter()
    - experiment_start
)

print()
print(
    f"Completed in "
    f"{elapsed:.2f} seconds"
)


ZOOM 23
boston-seaport: building index...
singapore-hollandvillage: building index...
singapore-onenorth: building index...
singapore-queenstown: building index...
Checkpoint written: /Users/abinavbala/College/Research/BST-BEV/outputs/direct_query_all_samples.csv

ZOOM 24
boston-seaport: building index...
singapore-hollandvillage: building index...
singapore-onenorth: building index...
singapore-queenstown: building index...
Checkpoint written: /Users/abinavbala/College/Research/BST-BEV/outputs/direct_query_all_samples.csv

Completed in 14.73 seconds


In [9]:
results_df.groupby("zoom").agg(
    samples=(
        "sample_token",
        "count",
    ),

    exact_candidate_matches=(
        "same_candidates",
        "sum",
    ),

    exact_cell_count_matches=(
        "same_cell_count",
        "sum",
    ),

    min_jaccard=(
        "candidate_jaccard",
        "min",
    ),

    mean_jaccard=(
        "candidate_jaccard",
        "mean",
    ),
)

,samples,exact_candidate_matches,exact_cell_count_matches,min_jaccard,mean_jaccard
zoom,,,,,
23,404,404,404,1.0,1.0
24,404,404,403,1.0,1.0


In [10]:
timing_summary = (
    results_df
    .groupby("zoom")
    .agg(
        mean_sampling_ms=(
            "sampling_time_ms",
            "mean",
        ),

        median_sampling_ms=(
            "sampling_time_ms",
            "median",
        ),

        mean_direct_ms=(
            "direct_total_time_ms",
            "mean",
        ),

        median_direct_ms=(
            "direct_total_time_ms",
            "median",
        ),

        mean_cover_ms=(
            "direct_cover_time_ms",
            "mean",
        ),

        mean_lookup_ms=(
            "direct_lookup_time_ms",
            "mean",
        ),

        mean_speedup=(
            "speedup",
            "mean",
        ),

        median_speedup=(
            "speedup",
            "median",
        ),

        mean_cells=(
            "direct_cells",
            "mean",
        ),
    )
    .reset_index()
)

timing_summary

,zoom,mean_sampling_ms,median_sampling_ms,mean_direct_ms,median_direct_ms,mean_cover_ms,mean_lookup_ms,mean_speedup,median_speedup,mean_cells
0,23,11.555671,11.31750,1.279999,1.048209,0.354782,0.925217,9.731855,11.034044,239.913366
1,24,12.728958,11.45425,4.686760,3.615958,1.237894,3.448867,2.865592,3.112854,901.287129


In [11]:
quality_summary = (
    results_df
    .groupby("zoom")
    .agg(
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        fn=("fn", "sum"),

        total_candidates=(
            "direct_candidates",
            "sum",
        ),

        total_ground_truth=(
            "ground_truth",
            "sum",
        ),

        mean_cells=(
            "direct_cells",
            "mean",
        ),
    )
    .reset_index()
)

quality_summary["precision"] = (
    quality_summary["tp"]
    / (
        quality_summary["tp"]
        + quality_summary["fp"]
    )
)

quality_summary["recall"] = (
    quality_summary["tp"]
    / (
        quality_summary["tp"]
        + quality_summary["fn"]
    )
)

quality_summary

,zoom,tp,fp,fn,total_candidates,total_ground_truth,mean_cells,precision,recall
0,23,4065,262,0,4327,4065,239.913366,0.939450,1.0
1,24,4065,173,0,4238,4065,901.287129,0.959179,1.0


In [12]:
summary = quality_summary.merge(
    timing_summary,
    on=[
        "zoom",
        "mean_cells",
    ],
)

summary

,zoom,tp,fp,fn,total_candidates,total_ground_truth,mean_cells,precision,recall,mean_sampling_ms,median_sampling_ms,mean_direct_ms,median_direct_ms,mean_cover_ms,mean_lookup_ms,mean_speedup,median_speedup
0,23,4065,262,0,4327,4065,239.913366,0.939450,1.0,11.555671,11.31750,1.279999,1.048209,0.354782,0.925217,9.731855,11.034044
1,24,4065,173,0,4238,4065,901.287129,0.959179,1.0,12.728958,11.45425,4.686760,3.615958,1.237894,3.448867,2.865592,3.112854


In [13]:
summary.to_csv(
    SUMMARY_OUTPUT,
    index=False,
)

print(
    "Raw:",
    RAW_OUTPUT,
)

print(
    "Summary:",
    SUMMARY_OUTPUT,
)

Raw: /Users/abinavbala/College/Research/BST-BEV/outputs/direct_query_all_samples.csv
Summary: /Users/abinavbala/College/Research/BST-BEV/outputs/direct_query_summary.csv


In [14]:
mismatches = results_df[
    ~results_df["same_candidates"]
]

print(
    "Candidate mismatches:",
    len(mismatches),
)

mismatches[
    [
        "zoom",
        "scene",
        "sample_token",
        "location",
        "sampling_cells",
        "direct_cells",
        "sampling_candidates",
        "direct_candidates",
        "candidate_jaccard",
    ]
].head(20)

Candidate mismatches: 0


,zoom,scene,sample_token,location,sampling_cells,direct_cells,sampling_candidates,direct_candidates,candidate_jaccard


In [15]:
cell_count_mismatches = results_df[
    ~results_df["same_cell_count"]
]

print(
    "Cell-count mismatches:",
    len(cell_count_mismatches),
)

Cell-count mismatches: 1


In [16]:
from bst_bev.query.hierarchical import (
    HierarchicalRegionNormalizer,
)

In [17]:
def test_hierarchical_compression(
    zoom,
    *,
    center_x,
    center_y,
    location,
    radius=30.0,
):
    # -----------------------------------------
    # Exact fixed-resolution cover
    # -----------------------------------------

    region_cover = FixedZoomRegionCover(
        location=location,
        zoom=zoom,
    )

    fixed = region_cover.cover(
        center_x=center_x,
        center_y=center_y,
        radius=radius,
    )

    # -----------------------------------------
    # Hierarchical normalization
    # -----------------------------------------

    normalizer = HierarchicalRegionNormalizer()

    hierarchical = normalizer.normalize(
        fixed.cells
    )

    # -----------------------------------------
    # Correctness
    # -----------------------------------------

    equivalent = normalizer.verify_equivalent(
        leaf_cells=fixed.cells,
        hierarchical_cells=hierarchical.cells,
        working_zoom=zoom,
    )

    return {
        "zoom": zoom,

        "fixed_cells": fixed.cell_count,

        "hierarchical_cells":
            hierarchical.hierarchical_cell_count,

        "compression_ratio":
            hierarchical.compression_ratio,

        "reduction_fraction":
            hierarchical.reduction_fraction,

        "equivalent": equivalent,

        "normalization_time_ms":
            hierarchical.normalization_time_ms,

        "zoom_distribution":
            hierarchical.zoom_distribution(),
    }

In [18]:
test_sample = samples_df.iloc[0]

test_ego_x = test_sample["ego_x"]
test_ego_y = test_sample["ego_y"]
test_location = test_sample["location"]

print("Scene:", test_sample["scene"])
print("Location:", test_location)
print("Ego:", test_ego_x, test_ego_y)

Scene: scene-0061
Location: singapore-onenorth
Ego: 411.3039349319818 1180.8903791765097


In [19]:
result_23 = test_hierarchical_compression(
    23,
    center_x=test_ego_x,
    center_y=test_ego_y,
    location=test_location,
    radius=RADIUS_M,
)

result_23

{'zoom': 23,
 'fixed_cells': 196,
 'hierarchical_cells': 12,
 'compression_ratio': 16.333333333333332,
 'reduction_fraction': 0.9387755102040817,
 'equivalent': True,
 'normalization_time_ms': 501.7255840357393,
 'zoom_distribution': {(20, 20): 1,
  (20, 21): 1,
  (20, 22): 1,
  (21, 20): 1,
  (21, 21): 1,
  (21, 22): 1,
  (23, 20): 2,
  (23, 21): 2,
  (23, 22): 2}}

In [20]:
result_24 = test_hierarchical_compression(
    24,
    center_x=test_ego_x,
    center_y=test_ego_y,
    location=test_location,
    radius=RADIUS_M,
)

result_24

{'zoom': 24,
 'fixed_cells': 702,
 'hierarchical_cells': 20,
 'compression_ratio': 35.1,
 'reduction_fraction': 0.9715099715099715,
 'equivalent': True,
 'normalization_time_ms': 21369.21550007537,
 'zoom_distribution': {(20, 20): 1,
  (20, 22): 2,
  (20, 23): 1,
  (20, 24): 1,
  (21, 20): 1,
  (21, 22): 2,
  (21, 23): 1,
  (21, 24): 1,
  (24, 20): 2,
  (24, 22): 4,
  (24, 23): 2,
  (24, 24): 2}}

In [21]:
result_24[
    "zoom_distribution"
]

{(20, 20): 1,
 (20, 22): 2,
 (20, 23): 1,
 (20, 24): 1,
 (21, 20): 1,
 (21, 22): 2,
 (21, 23): 1,
 (21, 24): 1,
 (24, 20): 2,
 (24, 22): 4,
 (24, 23): 2,
 (24, 24): 2}

In [22]:
from bst_id.region_algebra import semantic_equal

from bst_bev.query.direct_hierarchical import (
    DirectHierarchicalRegionCover,
)

In [23]:
test_sample = samples_df.iloc[0]

test_ego_x = float(
    test_sample["ego_x"]
)

test_ego_y = float(
    test_sample["ego_y"]
)

test_location = str(
    test_sample["location"]
)

In [24]:
def test_direct_hierarchical(
    zoom,
):
    fixed_generator = (
        FixedZoomRegionCover(
            location=test_location,
            zoom=zoom,
        )
    )

    direct_generator = (
        DirectHierarchicalRegionCover(
            location=test_location,
            zoom=zoom,
        )
    )

    # Exact fixed-resolution reference.
    fixed = fixed_generator.cover(
        center_x=test_ego_x,
        center_y=test_ego_y,
        radius=RADIUS_M,
    )

    # New direct hierarchical cover.
    hierarchical = (
        direct_generator.cover(
            center_x=test_ego_x,
            center_y=test_ego_y,
            radius=RADIUS_M,
        )
    )

    equivalent = semantic_equal(
        fixed.cells,
        hierarchical.cells,
        working_zoom={
            "x": zoom,
            "y": zoom,
        },
    )

    return {
        "zoom": zoom,

        "fixed_cells":
            fixed.cell_count,

        "hierarchical_cells":
            hierarchical.hierarchical_cell_count,

        "x_blocks":
            len(hierarchical.x_blocks),

        "y_blocks":
            len(hierarchical.y_blocks),

        "compression_ratio":
            hierarchical.compression_ratio,

        "reduction_fraction":
            hierarchical.reduction_fraction,

        "equivalent":
            equivalent,

        "direct_cover_time_ms":
            hierarchical.cover_time_ms,
    }

In [25]:
direct_23 = test_direct_hierarchical(
    23
)

direct_23

{'zoom': 23,
 'fixed_cells': 196,
 'hierarchical_cells': 12,
 'x_blocks': 4,
 'y_blocks': 3,
 'compression_ratio': 16.333333333333332,
 'reduction_fraction': 0.9387755102040817,
 'equivalent': True,
 'direct_cover_time_ms': 0.056625111028552055}

In [26]:
direct_24 = test_direct_hierarchical(
    24
)

direct_24

{'zoom': 24,
 'fixed_cells': 702,
 'hierarchical_cells': 20,
 'x_blocks': 4,
 'y_blocks': 5,
 'compression_ratio': 35.1,
 'reduction_fraction': 0.9715099715099715,
 'equivalent': True,
 'direct_cover_time_ms': 0.07216585800051689}

In [27]:
from bst_bev.maps.hierarchical_bst_map_index import (
    HierarchicalBSTMapIndex,
)

from bst_bev.query.direct_hierarchical import (
    DirectHierarchicalRegionCover,
)

from bst_bev.query.fixed_zoom import (
    FixedZoomRegionCover,
)

from bst_bev.maps.bst_map_index import (
    BSTMapIndex,
)

ModuleNotFoundError: No module named 'bst_bev.maps.hierarchical_bst_map_index'

In [ ]:
test_sample = samples_df.iloc[0]

test_ego_x = float(
    test_sample["ego_x"]
)

test_ego_y = float(
    test_sample["ego_y"]
)

test_location = str(
    test_sample["location"]
)

print(
    test_sample["scene"],
    test_location,
    test_ego_x,
    test_ego_y,
)

In [ ]:
MIN_ZOOM = 20
MAX_ZOOM = 24

In [ ]:
hierarchical_index = (
    HierarchicalBSTMapIndex(
        nusc_map=nusc_maps[
            test_location
        ],
        location=test_location,
        min_zoom=MIN_ZOOM,
        max_zoom=MAX_ZOOM,
        index_spacing_m=1.0,
    )
)

In [ ]:
hierarchical_index.build()

In [ ]:
print(
    "Hierarchical keys:",
    f"{hierarchical_index.key_count:,}",
)

print(
    "Primitives:",
    f"{hierarchical_index.primitive_count:,}",
)

print(
    "Geometry samples:",
    f"{hierarchical_index.index_sample_count:,}",
)

print(
    "Unique leaf memberships:",
    f"{hierarchical_index.unique_leaf_memberships:,}",
)

print(
    "Hierarchical memberships:",
    f"{hierarchical_index.membership_count:,}",
)

print(
    "Build time:",
    f"{hierarchical_index.build_time_s:.3f}s",
)

In [ ]:
def compare_fixed_and_hierarchical(
    zoom,
):
    # ====================================================
    # FIXED-RESOLUTION BASELINE
    # ====================================================

    fixed_index = BSTMapIndex(
        nusc_map=nusc_maps[
            test_location
        ],
        location=test_location,
        zoom=zoom,
        index_spacing_m=1.0,
        adaptive_index_spacing=False,
    )

    fixed_index.build()

    fixed_cover_generator = (
        FixedZoomRegionCover(
            location=test_location,
            zoom=zoom,
        )
    )

    start = time.perf_counter()

    fixed_cover = (
        fixed_cover_generator.cover(
            center_x=test_ego_x,
            center_y=test_ego_y,
            radius=RADIUS_M,
        )
    )

    fixed_cover_time_ms = (
        time.perf_counter()
        - start
    ) * 1000.0

    start = time.perf_counter()

    fixed_tokens = (
        fixed_index.query_cell_tokens(
            fixed_cover.keys
        )
    )

    fixed_lookup_time_ms = (
        time.perf_counter()
        - start
    ) * 1000.0

    fixed_total_time_ms = (
        fixed_cover_time_ms
        + fixed_lookup_time_ms
    )

    # ====================================================
    # HIERARCHICAL QUERY
    # ====================================================

    hierarchical_cover_generator = (
        DirectHierarchicalRegionCover(
            location=test_location,
            zoom=zoom,
        )
    )

    hierarchical_cover = (
        hierarchical_cover_generator.cover(
            center_x=test_ego_x,
            center_y=test_ego_y,
            radius=RADIUS_M,
        )
    )

    hierarchical_result = (
        hierarchical_index.query_cells(
            hierarchical_cover.cells
        )
    )

    hierarchical_tokens = (
        hierarchical_result.tokens
    )

    hierarchical_total_time_ms = (
        hierarchical_cover.cover_time_ms
        + hierarchical_result.lookup_time_ms
    )

    # ====================================================
    # CANDIDATE EQUIVALENCE
    # ====================================================

    intersection = (
        fixed_tokens
        & hierarchical_tokens
    )

    union = (
        fixed_tokens
        | hierarchical_tokens
    )

    jaccard = (
        len(intersection)
        / len(union)
        if union
        else 1.0
    )

    # ====================================================
    # GROUND TRUTH
    # ====================================================

    gt_tokens = get_ground_truth_tokens(
        nusc_map=nusc_maps[
            test_location
        ],
        ego_x=test_ego_x,
        ego_y=test_ego_y,
        radius=RADIUS_M,
    )

    tp = len(
        hierarchical_tokens
        & gt_tokens
    )

    fp = len(
        hierarchical_tokens
        - gt_tokens
    )

    fn = len(
        gt_tokens
        - hierarchical_tokens
    )

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    return {
        "zoom": zoom,

        # Region sizes.
        "fixed_cells":
            fixed_cover.cell_count,

        "hierarchical_cells":
            hierarchical_cover.hierarchical_cell_count,

        "compression_ratio":
            hierarchical_cover.compression_ratio,

        # Candidates.
        "fixed_candidates":
            len(fixed_tokens),

        "hierarchical_candidates":
            len(hierarchical_tokens),

        "same_candidates":
            fixed_tokens
            == hierarchical_tokens,

        "candidate_jaccard":
            jaccard,

        "fixed_only":
            fixed_tokens
            - hierarchical_tokens,

        "hierarchical_only":
            hierarchical_tokens
            - fixed_tokens,

        # Accuracy.
        "precision":
            precision,

        "recall":
            recall,

        "tp": tp,
        "fp": fp,
        "fn": fn,

        # Fixed timing.
        "fixed_cover_time_ms":
            fixed_cover_time_ms,

        "fixed_lookup_time_ms":
            fixed_lookup_time_ms,

        "fixed_total_time_ms":
            fixed_total_time_ms,

        # Hierarchical timing.
        "hierarchical_cover_time_ms":
            hierarchical_cover.cover_time_ms,

        "hierarchical_lookup_time_ms":
            hierarchical_result.lookup_time_ms,

        "hierarchical_total_time_ms":
            hierarchical_total_time_ms,

        # Overall query speedup.
        "query_speedup": (
            fixed_total_time_ms
            / hierarchical_total_time_ms
            if hierarchical_total_time_ms > 0
            else float("inf")
        ),
    }

In [ ]:
result_23 = (
    compare_fixed_and_hierarchical(
        23
    )
)

result_23

In [ ]:
result_24 = (
    compare_fixed_and_hierarchical(
        24
    )
)

result_24

In [ ]:
fixed_24_index = BSTMapIndex(
    nusc_map=nusc_maps[
        test_location
    ],
    location=test_location,
    zoom=24,
    index_spacing_m=1.0,
    adaptive_index_spacing=False,
)

fixed_24_index.build()

In [ ]:
fixed_memberships = sum(
    len(primitives)
    for primitives
    in fixed_24_index.index.values()
)

In [ ]:
storage_comparison = {
    "fixed_keys":
        len(fixed_24_index),

    "hierarchical_keys":
        hierarchical_index.key_count,

    "fixed_memberships":
        fixed_memberships,

    "hierarchical_memberships":
        hierarchical_index.membership_count,

    "key_multiplier": (
        hierarchical_index.key_count
        / len(fixed_24_index)
        if len(fixed_24_index)
        else 0
    ),

    "membership_multiplier": (
        hierarchical_index.membership_count
        / fixed_memberships
        if fixed_memberships
        else 0
    ),
}

storage_comparison

In [ ]:
import pandas as pd


step_3c_results = pd.DataFrame([
    result_23,
    result_24,
])

step_3c_results.to_csv(
    OUTPUT_DIR
    / "hierarchical_retrieval_single_sample.csv",
    index=False,
)

In [ ]:
pd.DataFrame([
    storage_comparison
]).to_csv(
    OUTPUT_DIR
    / "hierarchical_index_storage_single_map.csv",
    index=False,
)